# UC Berkeley Capstone Project -- Data Analysis & Modelling

This is a notebook for my UC Berkeley Capstone project.

I have a harvester script running in GCP that collects YouTube video metadata and metrics and stores that in a BQ table. 
Because the script is running in an ongoing basis, and I have limited time before I need to turn in the project, I am going to start working with synthetic data integrated with my real data, and gradually shift over to all real data.

- This will allow me start modelling sooner. 
- The synthetic data will be identifiable in the DataFrame.
- I will be storing a Parquet snapshot of the BQ data extract + model version in a GCS Bucket, so that I can always see which specific data + model produced which specific results.

The workflow here is:

1. Create a snapshot of the BQ data 
1. Add the synthetic data rows
1. Store the dataset in a versioned GCS bucket
1. (when modelling) store model vesrion data + stats in a versioned GCS bucket as well.

In [3]:
import pandas as pd
import numpy as np

import utils.snapshot_data as snapshot


In [4]:
# See what's available
snapshot.list_snapshots()

# Load an existing snapshot (no new version created)
initial_df, meta = snapshot.load_snapshot("v1.0_real")

# Only create a new snapshot when you intentionally want one
# initial_df, meta = snapshot.snapshot_training_data("v1.1_real", notes="Added 3 more days of data")

initial_df.head()

Found 1 snapshots:

  v1.0_real  |  12990 rows  |  2026-03-22
    Polls: {'upload': 5322, '24h': 5041, '7d': 2627}
    File:  gs://maduros-dolce-capstone-data/snapshots/snapshots_v1.0_real_12990rows_20260322_180157.parquet

Loaded snapshot 'v1.0_real': 12990 rows from 2026-03-22
  Polls: {'upload': 5322, '24h': 5041, '7d': 2627}


,video_id,poll_timestamp,channel_id,channel_handle,title,view_count,like_count,comment_count,face_count,brightness,...,description,tags,duration_seconds,category_id,category_name,published_at,poll_label,hours_since_publish,subscriber_count,contains_synthetic_media
0,n3BeLZHjszE,2026-03-22 10:05:19.229224+00:00,UCJ8clxRNNMOF-1KaRFVaLlQ,@aqibtechreview134,Samsung Galaxy S27 Ultra Official Trailer & OF...,27,1,0,0,59.873854,...,The Future is Here: Samsung Galaxy S27 Ultra O...,"[Samsung Galaxy S27 Ultra, Galaxy S27 Ultra Le...",654,28,Science & Technology,2026-03-22 06:44:57+00:00,upload,3.34,1610,False
1,-zFOL2-QpPM,2026-03-22 10:05:19.229224+00:00,UCJ8clxRNNMOF-1KaRFVaLlQ,@aqibtechreview134,"DJI Mini 6 Pro Official Leaks: 1-Inch Sensor, ...",3,0,0,0,51.670718,...,🚀 DJI Mini 6 Pro: The Future of Drone Technolo...,"[DJI Mini 6 Pro, Mini 6 Pro Leaks, DJI Mini 6 ...",706,28,Science & Technology,2026-03-22 05:40:56+00:00,upload,4.41,1610,False
2,DBwp3mPn-VQ,2026-03-22 10:05:19.229224+00:00,UCJ8clxRNNMOF-1KaRFVaLlQ,@aqibtechreview134,"Apple TV 2026: A18 Bionic, 8K Gaming & The $99...",8,0,0,0,62.892558,...,The living room war is officially over! Apple ...,"[Apple TV 2026 redesign, new Apple TV 4K 2026 ...",681,28,Science & Technology,2026-03-22 04:42:47+00:00,upload,5.38,1610,False
3,4sikDye-XP0,2026-03-22 10:05:19.229224+00:00,UCTMd_rwB7b3RZM6o6wc-KkQ,@bhavanashealthycooking,Indori Poha Ki Recipe. इंदौर के पोहा कैसे बना...,1621,22,0,1,45.844051,...,,[],1570,28,Science & Technology,2026-03-22 06:07:50+00:00,upload,3.96,1490,False
4,iovDZ-GNeyo,2026-03-22 10:05:19.229224+00:00,UCfUoif9Pj1fA-Yy2yqFRQpQ,@parivlog-bihar,मम्मा कि बिटिया को भुखु लगी है | New Vlog | Vl...,883,0,0,0,88.155376,...,मम्मा कि बिटिया को भुखु लगी है | New Vlog | Vl...,"[Pari Vlog, Pari Life Style, Vlog Video, Daily...",64,22,People & Blogs,2026-03-22 06:33:45+00:00,upload,3.53,199000,False


In [5]:
initial_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12990 entries, 0 to 12989
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   video_id                  12990 non-null  object             
 1   poll_timestamp            12990 non-null  datetime64[us, UTC]
 2   channel_id                12990 non-null  object             
 3   channel_handle            12990 non-null  object             
 4   title                     12990 non-null  object             
 5   view_count                12990 non-null  Int64              
 6   like_count                12990 non-null  Int64              
 7   comment_count             12990 non-null  Int64              
 8   face_count                12990 non-null  Int64              
 9   brightness                12990 non-null  float64            
 10  colorfulness              12990 non-null  float64            
 11  vertical       